# Assignment 9 — Vision Transformer versus CNN

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** GPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and fair comparison

Compare a pre-trained Vision Transformer (ViT-B/16) with a pre-trained
convolutional model (ResNet18) on the same CIFAR-10 subset. Both ImageNet
backbones are frozen; only a new ten-class head is trained.

A CNN builds spatial features with local kernels. ViT divides an image
into patches and relates them using self-attention. Accuracy, macro F1,
parameter count, and training time provide a balanced comparison.


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

weights = models.ViT_B_16_Weights.DEFAULT
transform = weights.transforms(crop_size=224, resize_size=224)
train_set = datasets.CIFAR10("data", train=True, download=True, transform=transform)
test_set = datasets.CIFAR10("data", train=False, download=True, transform=transform)
rng = np.random.default_rng(SEED)
train_loader = DataLoader(Subset(train_set, rng.choice(len(train_set), 8000, replace=False)),
                          batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(Subset(test_set, rng.choice(len(test_set), 2000, replace=False)),
                         batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
def make_model(kind):
    if kind == "CNN (ResNet18)":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 10)
        head = model.fc
    elif kind == "ViT-B/16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, 10)
        head = model.heads.head
    else:
        raise ValueError(kind)
    for p in model.parameters(): p.requires_grad = False
    for p in head.parameters(): p.requires_grad = True
    return model.to(device)

def fit_and_evaluate(kind, epochs=3):
    model = make_model(kind)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    start = time.perf_counter()
    for epoch in range(epochs):
        # Gradients still update the new head while the frozen backbone stays in inference mode.
        model.eval()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(); logits = model(images)
            loss = criterion(logits, labels); loss.backward(); optimizer.step()
        print(kind, "completed epoch", epoch + 1)
    seconds = time.perf_counter() - start

    model.eval(); truth, prediction = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            output = model(images.to(device)).argmax(1).cpu().numpy()
            prediction.extend(output); truth.extend(labels.numpy())
    return {
        "model": kind,
        "accuracy": accuracy_score(truth, prediction),
        "macro_f1": f1_score(truth, prediction, average="macro"),
        "parameters_m": sum(p.numel() for p in model.parameters()) / 1e6,
        "train_seconds": seconds,
    }

comparison = pd.DataFrame([
    fit_and_evaluate("CNN (ResNet18)"),
    fit_and_evaluate("ViT-B/16"),
])
display(comparison.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
comparison.plot(x="model", y=["accuracy", "macro_f1"], kind="bar",
                ylim=(0, 1), ax=axes[0])
axes[0].set_title("Predictive performance"); axes[0].set_ylabel("Score")
comparison.plot(x="model", y="train_seconds", kind="bar",
                color="coral", legend=False, ax=axes[1])
axes[1].set_title("Head-training time"); axes[1].set_ylabel("Seconds")
for ax in axes: ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


## Discussion

ViT usually requires more computation and benefits greatly from large-scale
pretraining. CNNs contain a useful locality bias and can be efficient on
smaller datasets. This short experiment compares frozen features, not the
absolute limit of either architecture. Fine-tuning, augmentation, and more
epochs may change the ranking.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
